In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import lightgbm as lgb

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error

In [4]:
#前処理したデータ
client_data = pd.read_csv("client_data.csv")
pre_data = pd.read_csv("pre_data.csv")

In [5]:
#Xとyに分割
client_X = client_data.drop(columns=["y"])
client_y = client_data["y"]

X = pre_data.drop(columns=["y"])
y = pre_data["y"]

In [6]:
client_X.head()

,datetime,client,close,price_am,weekday,price_pm
0,20151208,1,0,0.0,1,0.0
1,20141218,1,0,0.0,3,0.0
2,20160129,1,0,0.0,4,0.0
3,20151221,1,0,0.0,0,0.0
4,20150114,1,0,0.0,2,0.0


In [7]:
X.head()

,datetime,client,close,price_am,weekday,price_pm
0,20100811,0,0,0.0,2,0.0
1,20101121,0,0,0.0,6,0.0
2,20101230,0,0,0.0,3,0.0
3,20101025,0,0,0.0,0,0.0
4,20150514,0,0,0.0,3,0.0


In [10]:
# パラメータグリッドを定義
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# GridSearchCVを初期化
grid_search = GridSearchCV(estimator=RandomForestRegressor(), param_grid=param_grid, 
                           cv=5, n_jobs=-1, scoring='neg_mean_absolute_error')

# グリッドサーチを実行
grid_search_client = grid_search.fit(client_X, client_y)
grid_search_other = grid_search.fit(X, y)

# 最適なモデルとそのパラメータを取得
best_model_client = grid_search_client.best_estimator_
best_params_client = grid_search_client.best_params_
best_model_other = grid_search_other.best_estimator_
best_params_other = grid_search_other.best_params_

# 最適なパラメータを表示
print("Best Parameters client:", best_params_client)
print("Best Parameters client:", best_params_other)

Best Parameters client: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Parameters client: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


In [11]:
#評価
cv_scores_client = cross_val_score(grid_search_client.best_estimator_, client_X, client_y, cv=5, scoring="neg_mean_absolute_error")
cv_scores_other = cross_val_score(grid_search_other.best_estimator_, X, y, cv=5, scoring="neg_mean_absolute_error")

#評価の表示
print("Cross Validation Scores client:", cv_scores_client)
print("Cross Validation Scores other:", cv_scores_other)

Cross Validation Scores client: [-3.16779167e-01 -1.37500000e-04 -5.95333333e-02 -8.18333333e-03
 -0.00000000e+00]
Cross Validation Scores other: [-2.6535     -0.39102083 -1.33416071 -0.86813393 -0.        ]


In [8]:
# MAEの計算
mae = mean_absolute_error(y, y_pred)
print("MAE:", mae)

MAE: 0.31482916666666666


In [9]:
#モデルの保存
with open("Apple_model.pickle", mode="wb") as fp:
    pickle.dump(model, fp)